import optuna
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("xgboost_optuna")

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("lr", 1e-3, 1e-1, log=True),
    }
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        score = train_and_eval(params)
        mlflow.log_metric("val_auc", score)
        return score

with mlflow.start_run(run_name="study_2026_05_19"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50)
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_auc", study.best_value)